# ViSTKG-QA — Chuỗi C1 → C2 → C3 (Colab GPU)

Notebook CHỈ clone repo + cài đặt + gọi script trong `src/` — không viết
logic mô hình/huấn luyện riêng ở đây. **Đã được duyệt trước cả chuỗi
C1→C2→C3 — chạy bằng "Run All", KHÔNG cần dừng xác nhận giữa các giai
đoạn.** Notebook chỉ dừng thật khi có lỗi/NaN thật (script `train.py` tự
raise RuntimeError nếu >5% mẫu 1 epoch bị NaN — xem docs/DECISIONS.md —
khi đó Colab sẽ dừng cell tại chỗ lỗi, không chạy tiếp các cell sau).

Mỗi cell IN RÕ dòng bắt đầu bằng `==RESULT==` ở cuối — copy các dòng này
dán lại cho Claude để báo cáo tiến độ.

Thứ tự: **C1 smoke test** → **C2 grid search alpha rút gọn (5 epoch/điểm, 1 seed)**
→ chọn alpha tốt nhất tự động → **C3 huấn luyện đầy đủ (alpha tốt nhất, 3 seed, tới 20 epoch, early stop patience 3)**.

## 0. Thiết lập: clone repo, cài đặt, mount Drive, xác nhận GPU

In [ ]:
# Sửa URL repo thật trước khi chạy (git remote hiện tại của dự án)
!git clone https://github.com/<TO_BE_FILLED>/stkg_vietnamese.git
%cd stkg_vietnamese

In [ ]:
# Pin CHÍNH XÁC transformers==4.42.3 + peft==0.12.0 (xem docs/DECISIONS.md —
# xung đột phiên bản đã xác nhận thật khi tải Vintern-1B-v2 trên bản mới hơn)
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/ViSTKG-QA/checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
print(f'==RESULT== Drive mounted, checkpoint root = {CHECKPOINT_ROOT}')

In [ ]:
# Xác nhận GPU thật đang dùng — DỪNG Ở ĐÂY nếu không thấy GPU (Runtime > Change runtime type > GPU)
!nvidia-smi

## C1 — Smoke test 50 mẫu thật (ĐÃ DUYỆT TRƯỚC)

Đã PASS trên CPU local (50/50 step, 0 NaN, loss giảm 30.70→27.58, xem
docs/DECISIONS.md) — chạy lại ở đây trên GPU thật để xác nhận tốc độ +
không NaN/OOM trên GPU. Nếu cell này lỗi/NaN, DỪNG — không chạy C2/C3.

In [ ]:
!python -m src.train.smoke_test --set train.device=cuda
print('==RESULT== C1 smoke test: xem log phía trên — PASS nếu "Hoàn tất smoke test: 50/50 step thành công, 0 step NaN/lỗi" và "GIẢM".')

## C2 — Grid search alpha {0.1, 0.3, 0.5, 0.7, 0.9} (ĐÃ DUYỆT TRƯỚC)

Phiên bản rút gọn: 5 epoch/điểm, 1 seed. Tự động chọn alpha tốt nhất theo
val MRR, ghi vào results/training/best_alpha.txt — C3 đọc file này, KHÔNG
cần anh sửa tay giữa các cell.

In [ ]:
ALPHAS = [0.1, 0.3, 0.5, 0.7, 0.9]
for a in ALPHAS:
    !python -m src.train.train --seeds 42 --tag alpha_{a} \
        --set retrieval.alpha={a} \
        --set optimizer.max_epochs=5 \
        --set train.device=cuda \
        --set train.checkpoint_dir=/content/drive/MyDrive/ViSTKG-QA/checkpoints/alpha_grid_{a}

In [ ]:
!python -m src.train.select_best_alpha
with open('results/training/best_alpha.txt') as f:
    BEST_ALPHA = f.read().strip()
print(f'==RESULT== C2 hoàn tất. BEST_ALPHA = {BEST_ALPHA} (đã ghi results/training/best_alpha.txt, C3 sẽ tự đọc file này)')

## C3 — Huấn luyện đầy đủ, alpha tốt nhất (tự động từ C2), 3 seed (ĐÃ DUYỆT TRƯỚC)

Batch=32 (gradient accumulation), tới 20 epoch, early stop MRR patience 3.
Checkpoint tốt nhất mỗi seed lưu vào Drive.

In [ ]:
with open('results/training/best_alpha.txt') as f:
    BEST_ALPHA = f.read().strip()
print(f'Dùng BEST_ALPHA={BEST_ALPHA} từ C2')

!python -m src.train.train --seeds 42 43 44 --tag full_run \
    --set retrieval.alpha={BEST_ALPHA} \
    --set train.device=cuda \
    --set train.checkpoint_dir=/content/drive/MyDrive/ViSTKG-QA/checkpoints/full_run

In [ ]:
import json
with open('results/training/train_results_full_run.json', encoding='utf-8') as f:
    full_results = json.load(f)
print('==RESULT== C3 hoàn tất — kết quả 3 seed:')
for r in full_results:
    print(f"  seed={r['seed']}: best_val_mrr={r['best_val_mrr']:.4f}")
mean_mrr = sum(r['best_val_mrr'] for r in full_results) / len(full_results)
print(f'==RESULT== Trung bình 3 seed: val_MRR={mean_mrr:.4f}')
print('==RESULT== Checkpoint đã lưu tại /content/drive/MyDrive/ViSTKG-QA/checkpoints/full_run/seed{42,43,44}_best.pt')
print('==RESULT== C1->C2->C3 HOÀN TẤT. Copy toàn bộ các dòng ==RESULT== phía trên dán lại cho Claude.')

## (Chưa chạy tự động) C4 — Ablation, C5 — Eval đầy đủ

Chạy riêng sau khi xem kết quả C3, KHÔNG nằm trong chuỗi tự động ở trên
(C4/C5 dùng checkpoint từ `full_run`, cần xác nhận C3 ổn trước).

In [ ]:
# !python -m src.eval.run_ablations --seed 42 --execute
# !python -m src.eval.run_full_eval --checkpoint_dir /content/drive/MyDrive/ViSTKG-QA/checkpoints/full_run